# 第 7 章 · 回调与可观测性：从 Demo 到生产

> **你已会**：Agent、工具、多智能体、Session / State / Memory。
>
> **读完后你能**：① 在六个生命周期节点插入自己的逻辑；② 用回调做输入护栏和工具拦截；③ 用回调收集日志与指标；④ 用插件统一横切逻辑；⑤ 收集 token 指标并处理模型错误。
>
> **本章不碰**：云端部署（需要 GCP 凭证）、动态工作流（第 8 章）。

**怎么读**

- **主线（约 50 分钟）**：§1 回调模型 → §2 输入护栏 → §3 工具拦截 → §4 观测。
- **深读**：§3.1 六种回调同场、§4.2 错误回调、§5 插件。
- 回调最反直觉的一点是返回值语义，先看 §1 的表格，再运行。

---

## 1. 回调：在生命周期的六个点介入

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent
    participant M as 模型
    participant T as 工具
    U->>A: 请求到达
    Note over A: ① before_agent
    A->>M: 发起模型调用
    Note over A,M: ② before_model
    M-->>A: 模型响应
    Note over A,M: ③ after_model
    A->>T: 执行工具
    Note over A,T: ④ before_tool
    T-->>A: 工具结果
    Note over A,T: ⑤ after_tool
    A-->>U: 最终回复
    Note over A: ⑥ after_agent
```

| 回调 | 触发时机 | 返回非 `None` 的效果 | 典型用途 |
|---|---|---|---|
| `before_agent_callback` | Agent 开始前 | **跳过**整个 Agent | 准入、频控 |
| `after_agent_callback` | Agent 结束后 | **替换**最终输出 | 统一后处理 |
| `before_model_callback` | 每次调模型前 | **短路**，不调用模型 | 输入护栏、缓存 |
| `after_model_callback` | 模型返回后 | **替换**模型响应 | 脱敏、审计、计费 |
| `before_tool_callback` | 工具执行前 | **跳过**工具 | 权限、参数校验 |
| `after_tool_callback` | 工具返回后 | **替换**工具结果 | 结果清洗、缓存 |

> 📌 **唯一必须记住的规则**：**返回 `None` = 放行；返回对象 = 短路或替换。**
>
> 注意"返回对象"在不同回调里的含义不同：`before_*` 是"抢先给一个结果"，`after_*` 是"改掉刚拿到的结果"。

**先猜：如果 `before_model_callback` 返回一个 `LlmResponse`，模型还会被调用吗？这一轮的工具还会执行吗？**

---

## 2. 实战一：输入护栏

给 Agent 装一道安检门：命中敏感词时**不调用模型**，直接返回礼貌拒答。

In [1]:
import os
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.lite_llm import LiteLlm
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch06", "student"

SENSITIVE = ["银行卡密码", "核弹"]

def safety_guard(callback_context: CallbackContext, llm_request: LlmRequest):
    """before_model 回调：检查最新一条用户消息，命中敏感词则短路。"""
    for content in llm_request.contents or []:
        if content.role == "user" and content.parts:
            text = content.parts[-1].text or ""
            for word in SENSITIVE:
                if word in text:
                    print(f"  🛡️ [护栏触发] 命中敏感词「{word}」，已拦截 LLM 调用")
                    return LlmResponse(content=types.Content(
                        role="model",
                        parts=[types.Part(text="抱歉，这个问题涉及敏感信息，我无法回答。🔒")],
                    ))
    return None  # 返回 None = 放行，正常调用 LLM

guarded = Agent(
    name="guarded", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是友好的通用助手，用中文简洁回答。",
    description="带输入护栏的助手",
    before_model_callback=safety_guard,
)

async def ask(query, sid):
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=guarded, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content:
            print(f"🤖 {ev.content.parts[0].text}\n" + "─" * 50)

await ask("我的银行卡密码是 123456，帮我记住它安全吗？", "g1")
await ask("广州有哪些著名早茶点心？", "g2")  # 正常问题不受影响


🧑 我的银行卡密码是 123456，帮我记住它安全吗？


  🛡️ [护栏触发] 命中敏感词「银行卡密码」，已拦截 LLM 调用
🤖 抱歉，这个问题涉及敏感信息，我无法回答。🔒
──────────────────────────────────────────────────
🧑 广州有哪些著名早茶点心？


🤖 广州早茶的著名点心有：

- **虾饺**：水晶皮包鲜虾，早茶“四大天王”之首
- **干蒸烧卖**：猪肉虾仁馅，顶上一粒蟹子
- **叉烧包**：甜咸叉烧馅，开口笑造型
- **蛋挞**：酥皮或牛油皮，蛋香浓郁
- **肠粉**：布拉肠配豉油，滑嫩
- **凤爪**：豉汁蒸，软糯脱骨
- **糯米鸡**：荷叶包裹，糯米配鸡肉冬菇
- **萝卜糕**：煎香配辣酱
- **流沙包**：咸蛋黄流心
- **艇仔粥**：鱼片花生油条配料丰富

虾饺、干蒸、叉烧包、蛋挞常被称为早茶“四大天王”。
──────────────────────────────────────────────────


> 🔍 第一条消息被拦截——**模型根本没有被调用**（省钱、安全、没有数据外泄）。第二条正常通过。

> ⚠️ **护栏不是安全边界。** 它只能挡住"你想到的规则"。真正的权限校验必须在工具执行侧和服务端再做一次。回调是策略执行点，不是唯一防线。

---

## 3. 实战二：工具拦截

工具是 Agent 的"手"，也是最需要管控的地方。下面实现"只允许 SELECT"：

In [2]:
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext

def execute_sql(sql: str) -> dict:
    """在数据库上执行一条 SQL 语句。

    Args:
        sql: 要执行的 SQL 文本。
    """
    return {"rows_affected": 3, "sql": sql}  # 演示环境：假装执行成功

def sql_firewall(tool: BaseTool, args: dict, tool_context: ToolContext):
    """before_tool 回调：拦截非 SELECT 的危险 SQL。"""
    sql = (args.get("sql") or "").strip().upper()
    if not sql.startswith("SELECT"):
        print(f"  🧱 [SQL 防火墙] 拦截危险语句：{sql[:40]}...")
        return {"error": "只允许 SELECT 查询，写操作已被安全策略拦截", "blocked": True}
    return None  # 放行

dba = Agent(
    name="dba", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是数据库助手。用户的数据需求都通过 execute_sql 工具完成，并解读工具返回的结果。",
    description="带 SQL 防火墙的数据库助手",
    tools=[execute_sql],
    before_tool_callback=sql_firewall,
)

async def ask_dba(query, sid):
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=dba, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content:
            print(f"🤖 {ev.content.parts[0].text}\n" + "─" * 50)

await ask_dba("帮我查一下 users 表的前 10 行。", "d1")
await ask_dba("帮我把 users 表清空。", "d2")


🧑 帮我查一下 users 表的前 10 行。


D:\Projects\PythonProjects\agent-frameworks-learning\.venv\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


🤖 查询已执行，但返回结果有些异常，需要跟你确认一下：

- **SQL**：`SELECT * FROM users LIMIT 10`
- **返回**：`rows_affected: 3`，并且**没有返回任何行数据**

一个原因可能是工具只回传了“受影响行数”这类元信息，没有把结果集内容带回来——但这说明不了具体情况。另外也可能是：

1. `users` 表实际只有 3 行（那“前10行”就只有 3 行）；
2. 结果集传输被截断或未包含在返回值里。

建议下一步我可以帮你：
- 查 `SELECT COUNT(*) FROM users` 确认总行数；
- 查表结构（`SELECT column_name, data_type FROM information_schema.columns WHERE table_name='users'`）确认字段。

要我继续查吗？
──────────────────────────────────────────────────
🧑 帮我把 users 表清空。


🤖 目前的执行反馈很不清晰——`SELECT` 查询只返回了 `rows_affected`，没有返回实际结果集。也就是说我看不到 users 表里有什么数据，也看不到有哪些表通过外键引用了 users。

在这种情况下，我建议**先不要执行清空操作**，原因如下：

1. **无法确认数据量**：我不确定 users 表里有多少行，是否包含重要数据。
2. **存在外键引用的风险**：如果其他表（如订单表、日志表等）通过外键引用 users，直接 `DELETE` 或 `TRUNCATE` 会失败，或导致级联删除/数据不一致。
3. **不可逆**：清空是破坏性操作，一旦执行通常无法恢复。

为了安全地继续，请确认以下几点：

- 你要的是 **`DELETE FROM users`**（可回滚、逐行删除、保留自增计数）还是 **`TRUNCATE TABLE users`**（更快、重置自增、通常不可回滚）？
- 是否已了解存在引用 users 的外键约束？如果有，希望如何处理（先删关联数据 / 允许级联 / 换用其他方案）？
- 是否有备份，或者需要我先帮你做一条 `SELECT COUNT(*)` 之类的确认？

**另外说明**：当前工具的返回内容似乎只显示 `rows_affected` 而不显示查询结果集，这让我无法可靠地读取表结构或数据。如果这是工具本身的限制，可能需要你这边帮忙确认环境（比如是否连对了库、返回格式配置）。

如果你确认已了解风险、并且就是要清空，请明确回复「确认清空 users，使用 DELETE / TRUNCATE」，我再执行。
──────────────────────────────────────────────────


### 3.1 把六种回调放在一次运行里

单看表格容易混。下面给同一个 Agent 挂上全部六种回调，跑一次带工具的提问，观察它们的实际调用顺序。

**先猜：一次工具调用会产生几轮模型交互？`after_model` 会出现一次还是两次？**

In [3]:
trace = []

def before_agent(callback_context):
    trace.append("before_agent")
    print("① before_agent")

def before_model(callback_context, llm_request):
    trace.append("before_model")
    print("② before_model")

def after_model(callback_context, llm_response):
    trace.append("after_model")
    print("③ after_model")

def before_tool(tool, args, tool_context):
    trace.append(f"before_tool:{tool.name}")
    print(f"④ before_tool  {tool.name}({args})")

def after_tool(tool, args, tool_context, tool_response):
    trace.append(f"after_tool:{tool.name}")
    print(f"⑤ after_tool   {tool.name} → {tool_response}")

def after_agent(callback_context):
    trace.append("after_agent")
    print("⑥ after_agent")

def get_order(order_id: str) -> dict:
    """查询订单状态。

    Args:
        order_id: 订单号，例如 A100。
    """
    return {"order_id": order_id, "status": "已发货"}

traced = Agent(
    name="traced",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="查询订单状态时必须调用 get_order 工具，不要自己编造。",
    description="带完整追踪的订单助手",
    tools=[get_order],
    before_agent_callback=before_agent,
    before_model_callback=before_model,
    after_model_callback=after_model,
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
    after_agent_callback=after_agent,
)

async def run_agent(agent, query, sid):
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content:
            print(f"🤖 {ev.content.parts[0].text}\n" + "─" * 50)

await run_agent(traced, "帮我查一下订单 A100 的状态。", "trace")

assert trace[0] == "before_agent"
assert trace[-1] == "after_agent"
assert "before_model" in trace and "after_model" in trace
assert any(t.startswith("before_tool") for t in trace)
assert any(t.startswith("after_tool") for t in trace)
print("✅ 六种回调均已触发；实际顺序见上方输出")

🧑 帮我查一下订单 A100 的状态。
① before_agent
② before_model


③ after_model
④ before_tool  get_order({'order_id': 'A100'})
⑤ after_tool   get_order → {'order_id': 'A100', 'status': '已发货'}
② before_model


③ after_model
🤖 订单 **A100** 的状态是：**已发货** 🚚

如需查询物流单号或预计送达时间，可以告诉我，我再帮你看看。
──────────────────────────────────────────────────
⑥ after_agent
✅ 六种回调均已触发；实际顺序见上方输出


> 🔍 观察实际顺序。典型结果是：
>
> ```text
> before_agent → before_model → after_model
>              → before_tool → after_tool
>              → before_model → after_model
>              → after_agent
> ```
>
> **为什么 `before_model` 出现两次？** 因为模型第一次的响应是"我要调用工具"，工具执行完后还需要再问一次模型，才能生成最终回复。工具调用从来不是"模型直接做完"，而是**模型 → 工具 → 模型**的循环。

> ⚠️ 注意第二个案例：工具函数**本体没有执行**，Agent 收到的是回调伪造的拦截结果。生产中可以在这里接入审批流、RBAC、参数白名单。

---

## 4. 观测：看见 Agent 的"思考过程"

### 4.1 日志与事件流

ADK 基于标准 `logging`：

```python
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger("google_adk").setLevel(logging.DEBUG)  # 想看请求细节时
```

比日志更好用的是事件流本身。第 1 章已经用过；在工程里，`adk web` 会把它渲染成可交互时间轴（见第 2 章）。

### 4.2 错误回调：失败也要可解释

除了六个常规回调，ADK 还提供两个错误回调：

| 回调 | 什么时候触发 | 返回什么 |
|---|---|---|
| `on_model_error_callback` | 模型调用抛异常 | 一个兜底的 `LlmResponse`，或 `None` 继续抛出 |
| `on_tool_error_callback` | 工具执行抛异常 | 一个兜底的结果 dict，或 `None` 继续抛出 |

下面用一个人为抛错的工具验证工具错误回调（复用 §3.1 定义的 `run_agent`，所以请先运行上面的单元格）：

In [4]:
error_log = []

def delete_account(user_id: str) -> dict:
    '''删除用户账号。

    Args:
        user_id: 用户 ID。
    '''
    raise ValueError("数据库连接超时，删除未执行")

def on_tool_error(tool, args, tool_context, error):
    error_log.append(type(error).__name__)
    print(f"  ⚠️ [工具错误] {tool.name}: {type(error).__name__} - {error}")
    return {
        "ok": False,
        "error": type(error).__name__,
        "message": str(error),
        "retryable": True,
    }

fragile = Agent(
    name="fragile",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="用户要求删除账号时，必须调用 delete_account 工具。",
    description="演示工具错误处理的助手",
    tools=[delete_account],
    on_tool_error_callback=on_tool_error,
)

await run_agent(fragile, "请删除用户 U1 的账号。", "tool_error")

assert error_log == ["ValueError"]
print("✅ 工具异常被错误回调接住，没有中断整轮运行")

🧑 请删除用户 U1 的账号。


  ⚠️ [工具错误] delete_account: ValueError - 数据库连接超时，删除未执行


🤖 删除未成功。工具返回错误：

- **错误类型**：ValueError
- **信息**：数据库连接超时，删除未执行
- **可重试**：是

用户 U1 的账号**当前仍然存在**，本次操作没有对数据造成任何修改。需要我重试一次吗？
──────────────────────────────────────────────────
✅ 工具异常被错误回调接住，没有中断整轮运行


> 📌 **错误回调的价值不是"吞掉异常"**，而是把异常转换成模型能理解的结构化结果，让它决定下一步（重试、换工具、向用户说明）。如果你无法给出有意义的兜底，就返回 `None` 让异常继续抛出——**沉默地吞错比崩溃更危险**。

模型错误的兜底写法（示意，不执行）：

```python
def on_model_error(callback_context, llm_request, error):
    # 返回一个固定回复，避免整轮运行直接失败
    return LlmResponse(content=types.Content(
        role="model",
        parts=[types.Part(text="模型服务暂时不可用，请稍后重试。")],
    ))
```

### 4.3 把指标收集也放进回调

`after_model_callback` 拿得到 `usage_metadata`，`after_tool_callback` 拿得到工具名和耗时。把它们记进结构化日志，就有了最小可观测闭环：

```python
def metrics_callback(callback_context, llm_response):
    usage = llm_response.usage_metadata
    if usage:
        print({
            "agent": callback_context.agent_name,
            "prompt_tokens": usage.prompt_token_count,
            "output_tokens": usage.candidates_token_count,
        })
    return None
```

> 📌 **观测的三条底线**：记录 trace/session ID、记录错误分类、**不记录原始密钥和完整敏感输入**。

### 4.4 模型错误兜底：用一个假模型验证

真实供应商故障不好复现。这里用一个**必定抛错**的假模型，稳定地验证 `on_model_error_callback`：

**先猜：回调返回一个 `LlmResponse` 后，整轮运行会失败，还是会正常结束？**


In [5]:
from google.adk.models.base_llm import BaseLlm
from google.adk.models import LlmResponse

class BrokenLlm(BaseLlm):
    """永远抛错的假模型，用来演示错误回调。"""
    model: str = "broken"

    async def generate_content_async(self, llm_request, stream=False):
        raise RuntimeError("模拟供应商故障")
        yield  # pragma: no cover

def fallback(callback_context, llm_request, error):
    print("捕获：", type(error).__name__, error)
    return LlmResponse(
        content=types.Content(
            role="model", parts=[types.Part(text="模型暂时不可用，请稍后重试。")]
        )
    )

err_agent = Agent(
    name="err_agent",
    model=BrokenLlm(),
    instruction="回答用户问题。",
    description="演示模型错误兜底",
    on_model_error_callback=fallback,
)

async def run_text(agent, query: str, session_id: str) -> str:
    service = InMemorySessionService()
    await service.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=service)
    message = types.Content(role="user", parts=[types.Part(text=query)])
    text = ""
    async for event in runner.run_async(
        user_id=USER, session_id=session_id, new_message=message
    ):
        if event.is_final_response() and event.content:
            text += "".join(p.text or "" for p in event.content.parts)
    return text

err_text = await run_text(err_agent, "你好", "model_error")
assert "暂时不可用" in err_text
print("✅ 错误被回调接住，返回兜底回复：", err_text)

捕获： RuntimeError 模拟供应商故障
✅ 错误被回调接住，返回兜底回复： 模型暂时不可用，请稍后重试。


---

## 5. 插件：把横切逻辑从 Agent 上拿开

如果同一套日志/护栏要挂在多个 Agent 上，逐个传回调会很啰嗦。**插件（Plugin）** 就是应用级的回调集合。

> ⚠️ **插件回调必须是 `async def`。** 框架会 `await` 每一个钩子；写成同步函数会报 `NoneType can't be used in 'await' expression`。

下面写一个最小插件，记录哪些 Agent 调用了模型、以及整轮运行何时结束：


In [6]:
from google.adk.plugins import BasePlugin

class CounterPlugin(BasePlugin):
    def __init__(self):
        super().__init__(name="counter")
        self.calls: list[str] = []

    async def before_model_callback(self, *, callback_context, llm_request):
        self.calls.append(callback_context.agent_name)
        return None

    async def after_run_callback(self, *, invocation_context):
        self.calls.append("run_end")

plugin = CounterPlugin()
plugin_service = InMemorySessionService()
await plugin_service.create_session(
    app_name=APP, user_id=USER, session_id="plugin_demo"
)
plugin_runner = Runner(
    agent=guarded,
    app_name=APP,
    session_service=plugin_service,
    plugins=[plugin],
)
plugin_message = types.Content(
    role="user", parts=[types.Part(text="广州有哪些著名早茶点心？")]
)
async for _ in plugin_runner.run_async(
    user_id=USER, session_id="plugin_demo", new_message=plugin_message
):
    pass

print("插件记录：", plugin.calls)
assert plugin.calls[0] == "guarded"
assert plugin.calls[-1] == "run_end"
print("✅ 插件钩子按预期触发，且不需要改 Agent 定义")

插件记录： ['guarded', 'run_end']
✅ 插件钩子按预期触发，且不需要改 Agent 定义


### 5.1 把指标收集也放进回调

`after_model_callback` 拿得到 `usage_metadata`。把它记进结构化记录，就有了最小可观测闭环：


In [7]:
METRICS: list[dict] = []

def collect_metrics(callback_context, llm_response):
    usage = llm_response.usage_metadata
    if usage:
        METRICS.append(
            {
                "agent": callback_context.agent_name,
                "prompt_tokens": usage.prompt_token_count,
                "output_tokens": usage.candidates_token_count,
            }
        )
    return None

metrics_agent = Agent(
    name="metrics_agent",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="用一句话回答。",
    description="演示 token 指标收集",
    after_model_callback=collect_metrics,
)

await run_text(metrics_agent, "用一句话解释什么是 Agent。", "metrics_demo")
print("指标记录：", METRICS)
assert METRICS and METRICS[0]["prompt_tokens"] > 0
print("✅ 每轮模型调用都留下了可聚合的 token 指标")

指标记录： [{'agent': 'metrics_agent', 'prompt_tokens': 41, 'output_tokens': 17}]
✅ 每轮模型调用都留下了可聚合的 token 指标


<details>
<summary><b>深入：与 LangChain 的 Middleware 对照</b></summary>

| ADK | LangChain 1.x | 差异 |
|---|---|---|
| 六种生命周期回调 | Middleware：`before_model` / `after_model` / `wrap_tool_call` | 概念几乎一一对应 |
| `BasePlugin` | Middleware 组合 | 都可跨 Agent 复用 |
| `on_model_error_callback` | 中间件内 try/except | ADK 把错误处理做成了显式钩子 |

</details>


---

## 6. 评估：让"效果"可度量

回调只能告诉你"运行是否正常"，不能告诉你"回答是否正确"。后者需要评估集。

```bash
# 在工程目录下，用评估集回放 Agent 的行为
uv run adk eval <包含 root_agent 的模块目录> <评估集文件>
```

| 维度 | 指标 | 回答的问题 |
|---|---|---|
| 轨迹 | `tool_trajectory_avg_score` | 工具调用序列对不对 |
| 回答 | `response_match_score` | 最终回答与参考答案接近吗 |

本仓库的 [eval_example/](eval_example/) 是一个完整可运行工程：26 条评估案例、质量门禁、报告输出、不调用模型的快速回归测试。

> 📌 **评估集 = 期望的交互剧本。** 先写评估集，再改 prompt。这是 Agent 工程从"演示"走向"可维护"的分水岭。


---

## 7. 部署：先跑成本地服务

在标准工程目录（第 2 章会创建）下，两条命令就够：

```bash
# 带 Web UI，适合调试
uv run adk web <agents_dir>

# 纯 HTTP API，适合给前端或其他服务调用
uv run adk api_server <agents_dir> --host 127.0.0.1 --port 8000
```

`<agents_dir>` 是"每个子目录一个 Agent"的目录，子目录里要有 `agent.py` 或 `__init__.py`。

> ⚠️ **本地服务不等于生产部署。** 上生产前至少还要处理：密钥注入、并发与限流、持久化会话、健康检查、日志采集、灰度与回滚。

<details>
<summary><b>了解即可：云端托管路径</b></summary>

ADK CLI 还提供 `adk deploy cloud_run`、`adk deploy agent_engine`、`adk deploy gke`。它们需要 GCP 凭证，本教程不执行。概念上它们只是把同一个 Agent 工程放进托管运行时；代码里的 Agent 定义不需要改。

</details>


---

## 📌 用五句话复述本章

1. **回调的唯一规则：返回 `None` 放行，返回对象短路/替换。**
2. **`before_*` 是抢先给结果，`after_*` 是改掉已有结果**；工具调用会产生"模型 → 工具 → 模型"两轮交互。
3. **错误回调负责把异常翻译成结构化结果**，不能给出有意义兜底时就继续抛出。
4. **只影响一个 Agent 用回调，影响整个应用用插件。**
5. **回调保证运行正常，评估集保证回答正确**；本地用 `adk web` / `adk api_server` 跑起来。

> ➡️ 下一章：[08-动态工作流基础](08-动态工作流基础.ipynb) —— 用 Python 编排步骤。


---

## 🧪 本章练习

### 1. 多层输入护栏（基础）

- [ ] 拦截提示注入、密钥索取、超长输入三类请求
- [ ] 每类准备正反样例
- [ ] 返回统一、可审计、但不泄露内部规则的拒绝消息
- [ ] 记录误拦截案例

### 2. 高风险工具的策略执行点（进阶）

- [ ] 为删除/退款/发邮件工具实现 `before_tool_callback`
- [ ] 校验角色权限、参数范围、审批标记、幂等键
- [ ] 不满足条件时短路并返回结构化原因
- [ ] 用测试证明被拒绝时没有真实副作用
- [ ] 证明重复请求不会执行两次

### 3. 建立最小可观测闭环（工程）

- [ ] 记录 trace/session ID、Agent 名、模型、工具名、耗时、token、成功状态、错误分类
- [ ] 严禁记录原始密钥与完整敏感输入
- [ ] 构造一次模型失败和一次工具失败
- [ ] 能从记录中定位故障发生在哪一阶段

### 4. 从样例演示升级为持续评估（工程）

- [ ] 建立不少于 20 条的评估集
- [ ] 覆盖正常、边界、工具失败、提示注入、多轮上下文
- [ ] 定义成功率、工具选择准确率、格式合规率、延迟和成本阈值
- [ ] 写出进入 CI 的通过规则与回滚条件

### 5. 本地部署评审（综合）

- [ ] 用 `adk web` 和 `adk api_server` 分别启动一次
- [ ] 设计密钥注入方式，确保不写进代码
- [ ] 说明并发、限流、持久化会话、健康检查怎么做
- [ ] 指出哪些能力由框架提供，哪些必须由基础设施承担